# 01 — Google Colab T4 Setup: Ollama + DeepSeek-R1 + FastAPI Tunnel

This notebook provisions an isolated local LLM runtime on Google Colab's free T4 GPU tier:
1. Clones `daipayanghosal/llm-adversarial-threat-lab` and installs lightweight dependencies (`requirements.txt`).
2. Installs `ollama`, starts `ollama serve` in the background, and pulls `deepseek-r1:7b` (with automatic fallback to `deepseek-r1:1.5b`).
3. Smoke-tests `src/llm_client.py` (`<think>...</think>` trace separation).
4. Launches FastAPI (`src/api_server.py`) and exposes a public HTTPS tunnel via `cloudflared` (or `pyngrok`) for the React Security Workbench.

In [ ]:
# 1. Verify GPU, clone/sync repository, and install Python packages
!nvidia-smi || true
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/daipayanghosal/llm-adversarial-threat-lab.git'
REPO_DIR = Path('/content/llm-adversarial-threat-lab')

if not Path('src/api_server.py').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

!pip install -q -r requirements.txt
print('Workspace ready at:', Path.cwd())

In [ ]:
# 2. Install Ollama, launch daemon, and pull DeepSeek-R1 (7b with 1.5b fallback)
import shutil, subprocess, time, requests

if shutil.which('ollama') is None:
    !apt-get update -qq && apt-get install -y -qq zstd pciutils
    !curl -fsSL https://ollama.com/install.sh | sh

def _ollama_up():
    try:
        return requests.get('http://127.0.0.1:11434/api/version', timeout=1.5).status_code == 200
    except Exception:
        return False

if not _ollama_up():
    ollama_log = open('/tmp/ollama.log', 'a')
    subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(30):
        if _ollama_up():
            break
        time.sleep(1)

print('Ollama daemon ready on http://127.0.0.1:11434:', _ollama_up())

# Pull primary 7b distilled model; fall back to 1.5b if constrained
pull_rc = subprocess.run(['ollama', 'pull', 'deepseek-r1:7b']).returncode
if pull_rc != 0:
    print('Primary deepseek-r1:7b pull failed; falling back to deepseek-r1:1.5b')
    subprocess.run(['ollama', 'pull', 'deepseek-r1:1.5b'], check=True)

!ollama list

In [ ]:
# 3. Smoke-test OllamaClient + <think> parser
from src.llm_client import OllamaClient

client = OllamaClient()
print('Installed models:', client.list_models())
resp = client.generate(
    prompt='Briefly explain how TLS 1.3 prevents downgrade attacks.',
    system_prompt='You are a concise AI security assistant.',
    max_tokens=1024,
)
print('Active Model:', resp.model, '| Latency:', resp.latency_ms, 'ms')
print('--- <think> Trace ---\n', resp.thinking[:400])
print('--- Final Output ---\n', resp.final_response)

In [ ]:
# 4. Launch FastAPI Server + Cloudflare Quick Tunnel (no account required)
import re, threading, time, subprocess, requests, uvicorn
from pathlib import Path
from src.api_server import app

def _api_up():
    try:
        return requests.get('http://127.0.0.1:8000/health', timeout=1.5).status_code == 200
    except Exception:
        return False

if not _api_up():
    api_thread = threading.Thread(
        target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
        daemon=True,
    )
    api_thread.start()
    for _ in range(15):
        if _api_up():
            break
        time.sleep(0.5)

print('FastAPI /health active:', _api_up())

!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared

cf_log_path = Path('/tmp/cloudflared.log')
cf_log_file = open(cf_log_path, 'w')
subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=cf_log_file,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(45):
    if cf_log_path.exists():
        content = cf_log_path.read_text(errors='ignore')
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            public_url = match.group(0)
            break
    time.sleep(1)

if public_url:
    print('\n=== LIVE COLAB API TUNNEL READY ===')
    print('Paste into Workbench -> Live Colab API:', public_url)
else:
    print('Tunnel URL not yet visible. Inspect /tmp/cloudflared.log:')

In [ ]:
# Optional Alternative: pyngrok tunnel (uncomment if preferred over cloudflared)
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')
# print('Public ngrok URL:', ngrok.connect(8000).public_url)